# Notice
After cleaning the prior data, the last step needed before ML analysis is to represent the *semantic* meaning of the words via text embeddings.

We have done lexical analysis with pieces like URL length, but we need semantic meanings to be able to get at the root of our research question, to learn if semantic mining of information from IoCs can provide novel results.

Unfortunately, we cannot include the embedding step into this monolithic ipynb, as the cutting edge embeddings API we used ("gemini-embedding-001") is not freely available. We are able to use student credits for it, but not able to put it in a shared Colab our professor can run. To verify our work, you can look at the file `embed_data_pipeline.ipynb` in the repo, and that annotates more deeply how we got the embedded dataset for the above domains.

**Additionally! Cosine Similarity is an intensive process! 50,000 phishing URLs compared with 50,000 popular domains means 2,500,000,000 comparisons to be done. A local machine may run it faster than Google Colab, but it is our experience that it takes an hour or more to finish in Google Colab**

# Cosine Similarity Pipeline

This notebook combines two phases of the phishing-domain similarity analysis process:
1. **Part A - Computing Cosine Similarity** (`cosine_similarity.py`):  
   Loads PhishTank and Tranco embedding files, computes a cosine-similarity matrix, and saves the matrix to CSV. We generated embeddings data using the Google Gemini embeddings API and process those embeddings here.  
   Embeddings are useful for semantic IoC analysis because they map related indicators (e.g., lookalike domains, phishing-themed terms, or similar URL patterns) close together in vector space, helping detect meaningful relationships beyond exact string matches. We will analyze that vector space in ML after processing the embeddings.

2. **Part B - Extract Top-5 from a Pre-computed Matrix** (`top5_from_matrix.py`):  
   Streams the ~23GB of pre-existing matrix data (`FullMatrixOfCosineSimil.csv`) line-by-line and extract the top-5 most similar items per row **and** per column, keeping memory usage low. This matrix allows base level EDA and analysis to happen much easier.

---

## Imports

All libraries needed by both stages are loaded here.  

In [ ]:
# standard-library I/O
import csv
import json
import sys
import gc

# use min-heap for priority queue when choosing top-N matches
import heapq

# safe path handling across ipynb and Google resources
from pathlib import Path

# cosine similarity and other vector operations
import numpy as np

# used to easily export to CSV and handle dataframes
import pandas as pd

We have a requirement for this assignment that it run in Google Colab. As such, we pull the repo style and data from Github first.

In [ ]:
import os

# Clone the repo (skip if already present)
if not os.path.exists('DS_ML_Project_ColabIntegration'):
    !git clone https://github.com/ns15468-gasou/DS_ML_Project_ColabIntegration.git

os.chdir('DS_ML_Project_ColabIntegration/Project/notebooks')
print("Working directory:", os.getcwd())

Working directory: /content/DS_ML_Project_ColabIntegration/Project/notebooks


## Configuration

All our file paths and constants are defined in one place so they are easy to update.  
`BASE_DIR` is set to the directory containing this notebook for local running.
When embedding, I hit the limit of the 16GB of RAM on my machine, so the layout of the data is as follows:

embeds1 == 1st half of PhishTank Embeddings
embeds2 == 2nd half of PhishTank Embeddings
embedsA == All Tranco Embeddings

In [ ]:
BASE_DIR = Path.cwd().parent  # one level up from notebook cwd
DATA_DIR = BASE_DIR / "data"  # ../data

# Part A I/O
EMBEDS1_FILE = DATA_DIR / "embeds1.csv"
EMBEDS2_FILE = DATA_DIR / "embeds2.csv"
EMBEDSA_FILE = DATA_DIR / "embedsA.csv"
OUT_MATRIX   = DATA_DIR / "FullMatrixOfCosineSimil.csv"

# Part B I/O
FULL_MATRIX_FILE = DATA_DIR / "FullMatrixOfCosineSimil.csv"
OUT_ROW          = DATA_DIR / "top5_per_row.csv"
OUT_COL          = DATA_DIR / "top5_per_column.csv"

TOP_N = 5 # Used if we wanted to test more or fewer top matches

### Download Embedding Files from Cloud Storage

The three embedding CSVs are each multiple gigabytes, but we also have to allow the professor to run our data analysis.  
As such, they are streamed in 8 MB chunks so memory stays bounded during the download.  
Files that already exist on disk are skipped.

In [ ]:
import urllib.request
import shutil

_GCS_BASE = "https://storage.googleapis.com/gscsteam1-colabresultbucket"
_EMBED_FILES = {
    EMBEDS1_FILE: f"{_GCS_BASE}/embeds1.csv",
    EMBEDS2_FILE: f"{_GCS_BASE}/embeds2.csv",
    EMBEDSA_FILE: f"{_GCS_BASE}/embedsA.csv",
}

for local_path, url in _EMBED_FILES.items():
    if local_path.exists():
        print(f"  Already exists, skipping: {local_path.name}")
        continue
    print(f"  Downloading {local_path.name} …", flush=True)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url) as resp, open(local_path, "wb") as out:
        shutil.copyfileobj(resp, out, length=8 * 1024 * 1024)  # 8 MB chunks
    size_gb = local_path.stat().st_size / 1e9
    print(f"    Done ({size_gb:.2f} GB)")

print("All embedding files ready.")

  Already exists, skipping: embeds1.csv
  Already exists, skipping: embeds2.csv
  Already exists, skipping: embedsA.csv
All embedding files ready.


---
# Part A - Compute Cosine Similarity from Raw Embeddings

As mentioned earlier, Phishtank took up too much memory. As such, we combine 2 sets of their embeddings
(`embeds1` + `embeds2`), deduplicate them, and then compute their cosine similarity against a third set (`embedsA`).  
The outputted result is a full N×M matrix plus a handy top-5 summary.

### Helper Function - `load_embeddings`

Reads a CSV whose rows are `(embedding_json_array, text_label)`.  
Returns a dict `{text_label: embedding_vector}`, keeping only the first  
occurrence of each label.

This deduplicates the data by keeping a sort of running tally for rows.

In [ ]:
def load_embeddings(filepath: Path) -> dict[str, np.ndarray]:
    """Load a CSV embeddings file (embedding,text), returning {text: embedding_vector}.
    Vectors are stored as np.float32 arrays to cut memory ~7x vs Python lists."""
    result = {}
    with open(filepath, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)  # skip header row (embedding,text)
        for row in reader:
            if len(row) < 2:
                continue
            embedding_str, name = row[0], row[1]
            if name not in result:  # keep first occurrence
                result[name] = np.array(json.loads(embedding_str), dtype=np.float32)
    return result

### Helper - `build_matrix`

Converts the `{name: vector}` dictionary into a NumPy float32 matrix  
and the corresponding ordered list of names.  
This is needed so we can use fast vectorised operations for Cosine Similarity.
This approach was ellucidated from the work of Saidani, cited in our paper [5].

In [ ]:
def build_matrix(app_dict: dict[str, list[float]]) -> tuple[list[str], np.ndarray]:
    """Convert {name: vector} dict to (name_list, numpy_matrix)."""
    names = list(app_dict.keys())
    mat = np.array([app_dict[n] for n in names], dtype=np.float32)
    return names, mat

### Helper - `cosine_similarity_matrix`

Computes the pairwise cosine similarity between two sets of vectors  
using the efficient **normalise → matmul** approach:

$$
\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}
$$

By L2-normalising each row first, the dot product of the resulting  
unit vectors directly yields cosine similarity. This basic utilization of Cosine Similarity is taken from Sonowal and Gunikhan [6].

In [ ]:
def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between every row in A and every row in B.
    Returns shape (len(A), len(B)).
    """
    A_norm = A / np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A_norm @ B_norm.T

### Step A1 - Load & Deduplicate Embeddings

Load `embeds1.csv` and `embeds2.csv`, merge them (with `embeds1` "winning"
on name collisions), and then load `embedsA.csv` separately.

In [ ]:
print("Loading embeddings files")
emb1 = load_embeddings(EMBEDS1_FILE)
emb2 = load_embeddings(EMBEDS2_FILE)

# Merge: emb1 takes precedence; add only new keys from emb2
combined = {}
for name, vec in emb1.items():
    combined[name] = vec
for name, vec in emb2.items():
    if name not in combined:
        combined[name] = vec
del emb1, emb2  # free memory
gc.collect()

print(f"  Combined & deduplicated: {len(combined)} unique entries\n")

embA = load_embeddings(EMBEDSA_FILE)

Loading embeddings files
  Combined & deduplicated: 55625 unique entries



### Step A1.5 - Additional Data Cleaning & Deduplication of URLs

Before converting to NumPy arrays, we normalise every URL key by stripping
the scheme (`http://`, `https://`), a leading `www.`, and any trailing `/`.
Two URLs that differ only in these prefixes or suffixes will now map to the
same key, so we deduplicate one more time (keeping the first occurrence).

This is applied to **both** the merged PhishTank embeddings (`combined`)
and the Tranco embeddings (`embA`).

This additional data had some use during the EDA phase and embedding phase (as things like HTTP vs HTTPS can imply security differences) [4].
But, when aggregating for future ML, we just need the base level data.

In [ ]:
import re

def clean_url(url: str) -> str:
    """Normalise a URL key: strip scheme, leading www., and trailing slashes."""
    url = re.sub(r'^https?://', '', url)
    url = re.sub(r'^www\.', '', url)
    url = url.rstrip('/')
    return url

def deduplicate_cleaned(emb_dict: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    """Re-key an embeddings dict by cleaned URL, keeping the first occurrence."""
    cleaned: dict[str, np.ndarray] = {}
    for raw_key, vec in emb_dict.items():
        key = clean_url(raw_key)
        if key not in cleaned:
            cleaned[key] = vec
    return cleaned

before_combined = len(combined)
combined = deduplicate_cleaned(combined)
print(f"  combined: {before_combined} → {len(combined)} after URL cleaning")

before_A = len(embA)
embA = deduplicate_cleaned(embA)
print(f"  embA:     {before_A} → {len(embA)} after URL cleaning")

gc.collect()

  combined: 55625 → 54896 after URL cleaning
  embA:     50000 → 50000 after URL cleaning


0

### Step A2 - Build NumPy Arrays

Convert the Python dictionaries into dense float32 matrices  
to enable fast linear-algebra operations.

In [ ]:
print("Building numpy arrays")
names_12, mat_12 = build_matrix(combined)
del combined
gc.collect()

names_A, mat_A = build_matrix(embA)
del embA
gc.collect()

print(f"  Matrix 1&2: {mat_12.shape}  |  Matrix A: {mat_A.shape}")
mem_mb = (mat_12.nbytes + mat_A.nbytes) / 1e6
print(f"  Embedding memory: {mem_mb:.1f} MB")

Building numpy arrays
  Matrix 1&2: (54896, 3072)  |  Matrix A: (50000, 3072)
  Embedding memory: 1289.0 MB


### Step A3 – Compute Cosine Similarity (chunked, low-memory)

L2-normalise both matrices **in-place**, then multiply in small row-chunks
(`CHUNK_SIZE` rows at a time).  Each chunk is written straight to the output
CSV so the full N × M matrix never lives in RAM.

In [ ]:
CHUNK_SIZE = 500  # rows per chunk; keeps each chunk < 200 MB for 12 GB Colab

# L2-normalise in-place so we only need the dot product for cosine sim
norms_12 = np.linalg.norm(mat_12, axis=1, keepdims=True)
norms_12[norms_12 == 0] = 1.0
mat_12 /= norms_12
del norms_12

norms_A = np.linalg.norm(mat_A, axis=1, keepdims=True)
norms_A[norms_A == 0] = 1.0
mat_A /= norms_A
del norms_A
gc.collect()

n_rows = mat_12.shape[0]
print(f"Computing cosine similarity in {-(-n_rows // CHUNK_SIZE)} chunks "
      f"& streaming to CSV …", flush=True)

with open(OUT_MATRIX, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["App_Embeds12"] + names_A)

    for start in range(0, n_rows, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, n_rows)
        chunk_sim = mat_12[start:end] @ mat_A.T          # (chunk, M) float32
        for i, row_idx in enumerate(range(start, end)):
            writer.writerow(
                [names_12[row_idx]] + [f"{v:.6f}" for v in chunk_sim[i]]
            )
        del chunk_sim
        gc.collect()
        if start == 0 or (start // CHUNK_SIZE) % 20 == 0:
            print(f"  {end:,} / {n_rows:,} rows written", flush=True)

print(f"  Done – saved {OUT_MATRIX.name}")

Computing cosine similarity in 110 chunks & streaming to CSV …
  500 / 54,896 rows written
  10,500 / 54,896 rows written
  20,500 / 54,896 rows written
  30,500 / 54,896 rows written
  40,500 / 54,896 rows written
  50,500 / 54,896 rows written
  Done – saved FullMatrixOfCosineSimil.csv


### Step A4 – Verify Output

The full matrix was written during Step A3. This cell just confirms the file.

In [ ]:
size_gb = OUT_MATRIX.stat().st_size / 1e9
print(f"  File:  {OUT_MATRIX}")
print(f"  Size:  {size_gb:.2f} GB")
print(f"  Shape: ({len(names_12)}, {len(names_A)})")

# Free normalised matrices now that the CSV is written
del mat_12, mat_A
gc.collect()

  File:  /content/DS_ML_Project_ColabIntegration/Project/data/FullMatrixOfCosineSimil.csv
  Size:  24.71 GB
  Shape: (54896, 50000)


0

---
# Part B - Stream-Extract Top-5 from a Large Pre-computed Matrix

When the full similarity matrix already exists on disk  
(`FullMatrixOfCosineSimil.csv` at 20+ GB),  
we can't load it all into RAM.

Instead, this section **streams the file row-by-row** and maintains  
fixed-size min-heaps (size 5) to track the best matches:

| Output file | Answers the question |
|---|---|
| `top5_per_row.csv` | For each **row** item, which 5 column items are most similar? |
| `top5_per_column.csv` | For each **column** item, which 5 row items are most similar? |

Memory stays bounded: only the header row plus one heap (≤ 5 entries)  
per column are held in RAM at any time.

### Step B1 - Read the Matrix Header

Open the CSV just to read the first line.  
This gives us the column labels and lets us allocate one min-heap per column.

In [ ]:
print("Reading header …", flush=True)
with open(FULL_MATRIX_FILE, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)

# header[0] is the corner label (e.g. "App_Embeds12"), rest are column names
col_labels = header[1:]
num_cols = len(col_labels)
print(f"  {num_cols} columns detected.")

Reading header …
  50000 columns detected.


### Step B2 - Stream Rows & Collect Top-5 per Row and Column

For **each row** we maintain a min-heap of size `TOP_N` tracking  
the highest-similarity column items seen so far.  
Simultaneously, for **each column** we maintain a parallel min-heap  
tracking the highest-similarity row items.

Because a min-heap keeps the *smallest* element at the top,  
we can efficiently evict it whenever a larger value arrives,  
guaranteeing O(R · C · log N) total work with O(C · N) memory according to Datacamp [9].

In [ ]:
# Min-heaps for each column: heap of (similarity, row_label)
col_heaps: list[list[tuple[float, str]]] = [[] for _ in range(num_cols)]

print("Streaming rows …", flush=True)
row_results: list[tuple[str, list[tuple[float, str]]]] = []
row_count = 0

with open(FULL_MATRIX_FILE, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    next(reader)  # skip header

    for row in reader:
        row_label = row[0]
        values = row[1:]

        # --- top 5 for this row (min-heap of size TOP_N) ---
        row_pairs: list[tuple[float, str]] = []
        for j, val_str in enumerate(values):
            if j >= num_cols:
                break
            try:
                sim_val = float(val_str)
            except (ValueError, IndexError):
                continue

            # Row top-5
            if len(row_pairs) < TOP_N:
                heapq.heappush(row_pairs, (sim_val, col_labels[j]))
            elif sim_val > row_pairs[0][0]:
                heapq.heapreplace(row_pairs, (sim_val, col_labels[j]))

            # Column top-5
            heap = col_heaps[j]
            if len(heap) < TOP_N:
                heapq.heappush(heap, (sim_val, row_label))
            elif sim_val > heap[0][0]:
                heapq.heapreplace(heap, (sim_val, row_label))

        # Store row result sorted descending
        row_pairs.sort(reverse=True)
        row_results.append((row_label, row_pairs))

        row_count += 1
        if row_count % 5000 == 0:
            print(f"  {row_count:,} rows processed", flush=True)

print(f"  Done: {row_count:,} rows")

Streaming rows …
  5,000 rows processed
  10,000 rows processed
  15,000 rows processed
  20,000 rows processed
  25,000 rows processed
  30,000 rows processed
  35,000 rows processed
  40,000 rows processed
  45,000 rows processed
  50,000 rows processed
  Done: 54,896 rows


### Step B3 - Write Top-5 per Row

Each output row contains the item label, the names of its top-5 matches,  
and the corresponding similarity scores.

In [ ]:
print(f"Writing {OUT_ROW.name} …", flush=True)
with open(OUT_ROW, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(
        ["item"]
        + [f"match_{i+1}" for i in range(TOP_N)]
        + [f"score_{i+1}" for i in range(TOP_N)]
    )
    for row_label, pairs in row_results:
        names = [p[1] for p in pairs]
        scores = [f"{p[0]:.6f}" for p in pairs]
        while len(names) < TOP_N:
            names.append("")
            scores.append("")
        writer.writerow([row_label] + names + scores)

print(f"  Saved: {OUT_ROW}")

Writing top5_per_row.csv …
  Saved: /content/DS_ML_Project_ColabIntegration/Project/data/top5_per_row.csv


### Step B4 - Write Top-5 per Column

Same format, but now each row of the output represents one  
**column** item from the original matrix, along with the 5 row  
items that were most similar to it.

In [ ]:
print(f"Writing {OUT_COL.name} …", flush=True)
with open(OUT_COL, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(
        ["item"]
        + [f"match_{i+1}" for i in range(TOP_N)]
        + [f"score_{i+1}" for i in range(TOP_N)]
    )
    for j, col_label in enumerate(col_labels):
        pairs = sorted(col_heaps[j], reverse=True)
        names = [p[1] for p in pairs]
        scores = [f"{p[0]:.6f}" for p in pairs]
        while len(names) < TOP_N:
            names.append("")
            scores.append("")
        writer.writerow([col_label] + names + scores)

print(f"  Saved: {OUT_COL}")

Writing top5_per_column.csv …
  Saved: /content/DS_ML_Project_ColabIntegration/Project/data/top5_per_column.csv


### Part B Summary

In [ ]:
print("Part B complete")
print(f"  Rows streamed:  {row_count:,}")
print(f"  Columns:        {num_cols:,}")
print(f"\nOutputs:")
print(f"  → {OUT_ROW}")
print(f"  → {OUT_COL}")

Part B complete
  Rows streamed:  54,896
  Columns:        50,000

Outputs:
  → /content/DS_ML_Project_ColabIntegration/Project/data/top5_per_row.csv
  → /content/DS_ML_Project_ColabIntegration/Project/data/top5_per_column.csv


## Citations
[1] Gabriela Brezeanu, Alexandru Archip, and Codrut-Georgian Artene. Phish fighter: Self updating machine learning shield against phishing kits based on HTML code analysis. 13:4460–4486.

[2] Bibhu Dash and Meraj Farheen Ansari. An effective cybersecurity awareness training model: First defense of an organizational security strategy. International Research Journal of Engineering and Technology, 9:2395–0056, 04 2022.

[3] Dawn M. Sarno, Maggie W. Harris, and Jeffrey Black. Which phish is captured in the net? understanding phishing susceptibility and individual differences. 37(4):789–803._eprint:https://onlinelibrary.wiley.com/doi/pdf/10.1002/acp.4075.

[4] Giuseppe Desolda, Francesco Greco, and Luca Vigano. APOLLO: A GPT-based tool to detect phishing emails and generate explanations that warn users. 9(4):EICS003:1–EICS003:33.

[5] Nadjate Saidani, Kamel Adi, and Mohand Saïd Allili. A semantic-based classification approach for an enhanced spam detection. Computers & Security, 94, 2020-07.

[6] Sonowal and Gunikhan. Detecting phishing sms based on multiple correlation algorithms. SN Computer Science, 1, 11 2020.

[7] Panpan Zhang, Jing Ya, Tingwen Liu, Quangang Li, Jinqiao Shi, and Zhaojun Gu. imcircle: Automatic mining of indicators of compromise from the web. In 2019 IEEE  symposium on Computers and Communications (ISCC), 2019.

[8] Rasha Zieni, Luisa Massari, and Maria Carla Calzarossa. Phishing or not phishing? a survey on the detection of phishing websites. 11:18499–18519.

[9] Chugani, Vinod. “What Is Cosine Distance?” Datacamp.com, DataCamp, 28 July 2024, www.datacamp.com/tutorial/cosine-distance.

[10] Cisco Systems, Inc. PhishTank: Join the fight against phishing. phishtank.com, 2026.

[11] Victor Le Pochat, Tom Van Goethem, Samaneh Tajalizadehkhoob, Maciej Korczy´nski, and Wouter Joosen. Tranco: A Research-Oriented Top Sites Ranking Hardened Against Manipulation. In Proceedings of the 26th Annual Network and Distributed System Security Symposium (NDSS), 2019.